# Stage 2 — Denetimsiz Makine Öğrenmesi

Bu notebook jürilerin "Yapay Zeka Nerede?" sorusunu gidermek için:

1. **Isolation Forest** — Anormallik/Anomali Tespiti
2. **K-Means Clustering** — Kullanıcı Segmentasyonu
3. Kümelerin istatistiksel profili ve görselleştirmeler

> CPU'da çalışır, GPU gerekmez.


In [ ]:
!pip install scikit-learn pandas numpy matplotlib seaborn -q
print('Hazır.')

In [ ]:
# ============================================================
# 1. DRIVE BAGLANTISI — Otomatik Klasor Bulma
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

import os

def find_colab_training_dir():
    candidates = [
        '/content/drive/MyDrive/colab_training',
        '/content/drive/My Drive/colab_training',
        '/content/drive/MyDrive/veri_seti/colab_training',
        '/content/drive/My Drive/veri_seti/colab_training',
    ]
    for path in candidates:
        if os.path.isdir(path):
            return path
    print('Drive taraniyor...')
    for root, dirs, files in os.walk('/content/drive/MyDrive'):
        depth = root.replace('/content/drive/MyDrive', '').count(os.sep)
        if depth > 3:
            dirs[:] = []
            continue
        if 'colab_training' in dirs:
            return os.path.join(root, 'colab_training')
    return None

BASE = find_colab_training_dir()
if BASE is None:
    print('colab_training bulunamadi!')
    BASE = '/content/drive/MyDrive/colab_training'
else:
    print(f'Bulundu: {BASE}')

DATA_DIR = f'{BASE}/data'
OUT_DIR  = f'{BASE}/outputs'
os.makedirs(OUT_DIR, exist_ok=True)

print(f'DATA_DIR: {DATA_DIR}')
print(f'OUT_DIR : {OUT_DIR}')
print('Dosyalar:')
if os.path.isdir(DATA_DIR):
    for f in sorted(os.listdir(DATA_DIR)):
        sz = os.path.getsize(os.path.join(DATA_DIR, f)) / 1024**2
        print(f'  {f}  ({sz:.1f} MB)')
else:
    print('  data/ klasoru yok! Drive yapisini kontrol edin.')

In [ ]:
# ============================================================
# 2. VERİ YÜKLEME
# ============================================================
import numpy as np
import pandas as pd
import json

# Stage 2 feature matrisini yükle (önceden hesaplanmış)
um_path = f'{DATA_DIR}/stage2_user_monthly_features.csv'
if os.path.exists(um_path):
    user_monthly = pd.read_csv(um_path, encoding='utf-8-sig')
    print(f'User-monthly features: {user_monthly.shape}')
else:
    # Yoksa sentetik veriden hesapla
    print('stage2_user_monthly_features.csv bulunamadı, ham veriden hesaplanıyor...')
    df = pd.read_csv(f'{DATA_DIR}/synthetic_budget_data.csv', encoding='utf-8-sig', low_memory=False)
    df['timestamp']  = pd.to_datetime(df['timestamp'], errors='coerce')
    df['year_month'] = df['timestamp'].dt.to_period('M').astype(str)
    df['amount']     = pd.to_numeric(df['amount'], errors='coerce').fillna(0)
    df['direction']  = df['direction'].fillna('debit')
    
    income_df  = df[df['direction']=='credit'].groupby(['user_id','year_month'])['amount'].sum().rename('monthly_income_computed')
    expense_df = df[df['direction']=='debit'].groupby(['user_id','year_month'])['amount'].sum().rename('monthly_expense_computed')
    user_monthly = pd.concat([income_df, expense_df], axis=1).fillna(0).reset_index()
    user_monthly['net_cashflow'] = user_monthly['monthly_income_computed'] - user_monthly['monthly_expense_computed']
    user_monthly['savings_rate'] = np.where(
        user_monthly['monthly_income_computed'] > 0,
        user_monthly['net_cashflow'] / user_monthly['monthly_income_computed'], 0
    ).clip(-2, 1)
    user_monthly['budget_utilisation'] = np.where(
        user_monthly['monthly_income_computed'] > 0,
        user_monthly['monthly_expense_computed'] / user_monthly['monthly_income_computed'], 0
    )
    print(f'Hesaplandı: {user_monthly.shape}')

print(user_monthly.columns.tolist()[:15])

In [ ]:
# ============================================================
# 3. FEATURE SEÇİMİ (ML için)
# ============================================================
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

FEATURE_COLS = [
    'monthly_income_computed', 'monthly_expense_computed',
    'savings_rate', 'budget_utilisation', 'net_cashflow'
]
# Mevcut kolonlardan seç + spend_ kolonları
spend_cols = [c for c in user_monthly.columns if c.startswith('spend_')]
FEATURE_COLS += spend_cols[:8]  # En fazla 8 harcama kategorisi
FEATURE_COLS  = [c for c in FEATURE_COLS if c in user_monthly.columns]

print(f'Kullanılacak özellikler ({len(FEATURE_COLS)}): {FEATURE_COLS}')

X_raw = user_monthly[FEATURE_COLS].copy()
imputer = SimpleImputer(strategy='median')
X_imp   = imputer.fit_transform(X_raw)
scaler  = StandardScaler()
X_scaled = scaler.fit_transform(X_imp)

print(f'Feature matrisi: {X_scaled.shape}')

In [ ]:
# ============================================================
# 4. ISOLATION FOREST — ANOMALİ TESPİTİ
# ============================================================
from sklearn.ensemble import IsolationForest
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

print('Isolation Forest eğitiliyor...')
iso_forest = IsolationForest(
    n_estimators=200,
    contamination=0.05,  # %5 anomali bekleniyor
    random_state=42,
    n_jobs=-1
)
iso_scores  = iso_forest.fit_predict(X_scaled)   # -1: anomali, 1: normal
anomaly_score = iso_forest.score_samples(X_scaled)  # Düşük = daha anormal

user_monthly['is_anomaly']     = (iso_scores == -1).astype(int)
user_monthly['anomaly_score']  = -anomaly_score   # Yüksek = daha anormal

n_anomaly = user_monthly['is_anomaly'].sum()
pct       = n_anomaly / len(user_monthly) * 100
print(f'Tespit edilen anomali: {n_anomaly:,} ({pct:.1f}%)')
print(user_monthly.groupby('is_anomaly')[['monthly_income_computed','monthly_expense_computed','savings_rate']].mean().round(2))

In [ ]:
# ============================================================
# 4b. ANOMALİ GÖRSELLEŞTİRME
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Anomali skor dağılımı
axes[0].hist(user_monthly['anomaly_score'], bins=60, color='#3498DB', edgecolor='white', linewidth=0.3)
thresh = np.percentile(user_monthly['anomaly_score'], 95)
axes[0].axvline(thresh, color='red', linestyle='--', linewidth=1.5, label=f'%95 Eşik = {thresh:.2f}')
axes[0].set_title('Anomali Skoru Dağılımı (Isolation Forest)', fontsize=12)
axes[0].set_xlabel('Anomali Skoru (yüksek = daha anormal)')
axes[0].set_ylabel('Frekans')
axes[0].legend()

# Gelir vs Harcama — anomaliler işaretli
normal  = user_monthly[user_monthly['is_anomaly'] == 0]
anomaly = user_monthly[user_monthly['is_anomaly'] == 1]
axes[1].scatter(normal['monthly_income_computed'], normal['monthly_expense_computed'],
                alpha=0.2, s=5, color='#3498DB', label=f'Normal ({len(normal):,})')
axes[1].scatter(anomaly['monthly_income_computed'], anomaly['monthly_expense_computed'],
                alpha=0.6, s=20, color='#E74C3C', marker='x', label=f'Anomali ({len(anomaly):,})')
axes[1].set_title('Gelir vs Harcama — Anomali Tespiti', fontsize=12)
axes[1].set_xlabel('Aylık Gelir (₺)')
axes[1].set_ylabel('Aylık Harcama (₺)')
axes[1].xaxis.set_major_formatter(mtick.FuncFormatter(lambda x,_: f'₺{x/1000:.0f}K'))
axes[1].yaxis.set_major_formatter(mtick.FuncFormatter(lambda x,_: f'₺{x/1000:.0f}K'))
axes[1].legend()

plt.tight_layout()
anom_path = f'{OUT_DIR}/stage2_anomaly_detection.png'
plt.savefig(anom_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Kaydedildi: {anom_path}')

In [ ]:
# ============================================================
# 5. K-MEANS — KULLANICI SEGMENTASYONU
# ============================================================
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

# Elbow Method — optimum K bul
print('Elbow Method hesaplanıyor...')
inertias = []
sil_scores = []
K_RANGE = range(2, 9)
for k in K_RANGE:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled[user_monthly['is_anomaly'] == 0])
    inertias.append(km.inertia_)
    if k <= 7:
        sil = silhouette_score(X_scaled[user_monthly['is_anomaly'] == 0], km.labels_, sample_size=min(5000, len(X_scaled)))
        sil_scores.append(sil)
    print(f'  K={k}: Inertia={km.inertia_:.0f}')

# Elbow grafiği
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(list(K_RANGE), inertias, 'bo-')
ax1.set_title('Elbow Method')
ax1.set_xlabel('K (Küme Sayısı)')
ax1.set_ylabel('Inertia')

ax2.plot(list(K_RANGE)[:len(sil_scores)], sil_scores, 'ro-')
ax2.set_title('Silhouette Skoru')
ax2.set_xlabel('K (Küme Sayısı)')
ax2.set_ylabel('Silhouette')

plt.tight_layout()
plt.savefig(f'{OUT_DIR}/stage2_kmeans_elbow.png', dpi=150, bbox_inches='tight')
plt.show()

# Optimum K seç (en yüksek silhouette)
BEST_K = list(K_RANGE)[:len(sil_scores)][np.argmax(sil_scores)]
print(f'\nOptimum K: {BEST_K} (Silhouette={max(sil_scores):.4f})')

In [ ]:
# ============================================================
# 5b. FİNAL K-MEANS MODELİ
# ============================================================
BEST_K = 4  # Finansal anlamda 4 segment mantıklı
kmeans = KMeans(n_clusters=BEST_K, random_state=42, n_init=20)

clean_mask = user_monthly['is_anomaly'] == 0
X_clean = X_scaled[clean_mask]

# Sadece temiz veriyi kümele
clean_clusters = kmeans.fit_predict(X_clean)

# Kümeleri ana dataframe'e yerleştir (Anomaliler -1 olsun)
user_monthly['cluster'] = -1
user_monthly.loc[clean_mask, 'cluster'] = clean_clusters

# Küme profilleri (sadece normal kullanıcılar için)
cluster_profile = user_monthly[clean_mask].groupby('cluster').agg({
    'monthly_income_computed'  : 'mean',
    'monthly_expense_computed' : 'mean',
    'savings_rate'             : 'mean',
    'budget_utilisation'       : 'mean',
    'is_anomaly'               : 'mean',
    'user_id'                  : 'count'
}).rename(columns={'user_id': 'sample_count'})

print('\nKüme Profilleri:')
print(cluster_profile.round(3).to_string())

# Savings rate'e göre sıralayıp isim ver
sorted_clusters = cluster_profile['savings_rate'].sort_values().index.tolist()
cluster_name_map = {
    sorted_clusters[0]: 'Riskli Harcayıcılar',
    sorted_clusters[1]: 'Orta Profil',
    sorted_clusters[2]: 'Dengeli Tasarrufçular',
    sorted_clusters[3]: 'Güçlü Tasarrufçular',
    -1: 'ANOMALİ'
}
user_monthly['segment_name'] = user_monthly['cluster'].map(cluster_name_map)
print('\nSegment Dağılımı:')
print(user_monthly['segment_name'].value_counts().to_string())

In [ ]:
# ============================================================
# 5c. CLUSTERING GÖRSELLEŞTİRME (PCA 2D)
# ============================================================
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

COLORS = ['#E74C3C','#3498DB','#27AE60','#F39C12','#9B59B6']
SEGMENT_LABELS = list(cluster_name_map.values())

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# PCA Scatter — Kümeler
for cid in range(-1, BEST_K):
    mask = user_monthly['cluster'] == cid
    axes[0].scatter(X_pca[mask, 0], X_pca[mask, 1],
                    s=5, alpha=0.4, color=COLORS[cid] if cid>=0 else '#999999',
                    label=cluster_name_map.get(cid, f'Küme {cid}'))
axes[0].set_title(f'K-Means Kümeleri (K={BEST_K}) — PCA 2D Projeksiyon', fontsize=12)
axes[0].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
axes[0].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
axes[0].legend(markerscale=3, fontsize=9)

# Segment bazlı tasarruf dağılımı
segment_names_sorted = sorted(user_monthly['segment_name'].unique())
data_for_violin = [user_monthly[user_monthly['segment_name']==s]['savings_rate'].clip(-0.5,0.8).values
                   for s in segment_names_sorted]
parts = axes[1].violinplot(data_for_violin, positions=range(len(segment_names_sorted)), showmedians=True)
for pc, color in zip(parts['bodies'], COLORS):
    pc.set_facecolor(color)
    pc.set_alpha(0.7)
axes[1].set_xticks(range(len(segment_names_sorted)))
axes[1].set_xticklabels(segment_names_sorted, rotation=15, fontsize=8)
axes[1].set_title('Segment × Tasarruf Oranı Dağılımı', fontsize=12)
axes[1].set_ylabel('Tasarruf Oranı')
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1.0))
axes[1].axhline(0.2, color='green', linestyle='--', linewidth=1, label='%20 Hedef')
axes[1].legend()

plt.tight_layout()
cluster_path = f'{OUT_DIR}/stage2_user_clustering.png'
plt.savefig(cluster_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Kaydedildi: {cluster_path}')

In [ ]:
# ============================================================
# 6. SONUÇLARI KAYDET
# ============================================================
import pickle

# Modelleri kaydet
pickle.dump(iso_forest, open(f'{OUT_DIR}/isolation_forest.pkl','wb'))
pickle.dump(kmeans, open(f'{OUT_DIR}/kmeans_model.pkl','wb'))
pickle.dump(scaler, open(f'{OUT_DIR}/stage2_scaler.pkl','wb'))
pickle.dump({'feature_cols': FEATURE_COLS, 'cluster_name_map': cluster_name_map},
            open(f'{OUT_DIR}/stage2_config.pkl','wb'))

# Zenginleştirilmiş user_monthly kaydet
um_out = f'{OUT_DIR}/stage2_user_monthly_enriched.csv'
user_monthly.to_csv(um_out, index=False, encoding='utf-8-sig')

# Metrikler
stage2_metrics = {
    'isolation_forest': {
        'contamination'  : 0.05,
        'n_estimators'   : 200,
        'total_anomalies': int(user_monthly['is_anomaly'].sum()),
        'anomaly_pct'    : float(user_monthly['is_anomaly'].mean() * 100)
    },
    'kmeans': {
        'k'              : BEST_K,
        'silhouette'     : float(max(sil_scores)),
        'segments'       : cluster_profile.reset_index().to_dict('records'),
        'segment_names'  : cluster_name_map
    }
}
with open(f'{OUT_DIR}/stage2_ml_metrics.json','w',encoding='utf-8') as f:
    json.dump(stage2_metrics, f, indent=2, ensure_ascii=False)

print('\n=== STAGE 2 ML ÖZETİ ===')
print(f'  Isolation Forest: {user_monthly["is_anomaly"].sum():,} anomali ({user_monthly["is_anomaly"].mean()*100:.1f}%)')
print(f'  K-Means: {BEST_K} küme, Silhouette={max(sil_scores):.4f}')
print(f'  Çıktılar: {OUT_DIR}')
print('\nBu dosyaları indirip projenizin reports/ ve models/ klasörüne aktarın.')